In [2]:
import json
import numpy as np
import pandas as pd
import altair as alt

In [3]:
"""
analyze_taa_sensitivity.py
Analyzes how TAA parameter sensitivity relates to video characteristics (SI, TI, etc.)
Paste into a Jupyter notebook and run cell by cell.

Dependencies:
    pip install pandas altair numpy
"""

# ──────────────────────────────────────────────
# Config
# ──────────────────────────────────────────────

QUALITY_JSON    = "dataset.json"
VIDEO_STATS_CSV = "video_stats.csv"

RESOLUTIONS = ['50', '71', '87', '100']
PARAMS      = ['alpha_weight', 'hist_percent', 'filter_size', 'num_samples']
VIDEO_FEATURES = ["SI", "TI", "CF", "TP", "MV", "DTP"]

EXCLUDE_SCENES = [
    'junkyard-mound1', 'junkyard-mound2',
    'oldmine-speed-18', 'oldmine-speed-35',
    'oldmine-speed-75', 'oldmine-speed-9', 'oldmine-warm', 
    'wildwest-barzoom'
]

In [5]:
# Load in data
with open('../dataset.json', 'r') as f:
    raw = json.load(f)

records = []
for scene, scene_data in raw.items():
    if not isinstance(scene_data, dict):
        continue
    for res in RESOLUTIONS:
        if res not in scene_data:
            continue
        for ref, ref_data in scene_data[res].items():
            if ref != f'ref-{scene}': # we only want data when lower resolutions are compared to full-quality 16xSSAA
                continue
            for param in PARAMS:
                if param not in ref_data:
                    continue
                for entry in ref_data[param]:
                    records.append({
                        'scene':            scene,
                        'resolution':       int(res),
                        'parameter':        param,
                        'value':            entry['value'],
                        'score':            entry['score'],
                        'per_frame_errors': entry.get('per_frame_errors', []), # this is optional, can ignore
                    })

df = pd.DataFrame(records)
df = df[~df['scene'].isin(EXCLUDE_SCENES)]

# Exclude hist_percent values below 100 (must do because Unreal treats all sub 100 values as 100)
df = df[~((df['parameter'] == 'hist_percent') & (df['value'] < 100))]


### ________ Optional: Center Data _________
# find which values are shared across all scenes for each (parameter, resolution) 
common_vals_count = df.groupby(['parameter', 'resolution', 'value'])['scene'].nunique()
n_scenes_per_param_res = df.groupby(['parameter','resolution'])['scene'].nunique()

def center_score(group):
	# group is a subset of df for one specific (scene, parameter, resolution) combination 
	# e.g. all rows for "abandoned" + "alpha_weight" + resolution 50
    param, res = group['parameter'].iloc[0], group['resolution'].iloc[0]
    
    # get the values tested for this (parameter, resolution) across all scenes
    common = df[(df['parameter'] == param) & (df['resolution'] == res)]
    common_vals = common.groupby('value')['scene'].nunique()
    n_scenes = common['scene'].nunique()
    shared_vals = common_vals[common_vals == n_scenes].index
    
    # compute mean score over shared values only, then subtract from all scores
    mask = group['value'].isin(shared_vals)
    mean = group.loc[mask, 'score'].mean()
    return group['score'] - mean

df['score_centered'] = df.groupby(
    ['scene', 'parameter', 'resolution'], group_keys=False
).apply(center_score)

/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_12904/2734461361.py:58: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(center_score)


In [6]:
print(f"Quality data: {len(df)} rows, {df['scene'].nunique()} scenes")

Quality data: 2240 rows, 20 scenes


In [7]:
#  Load video stats

stats_df = pd.read_csv(VIDEO_STATS_CSV)
stats_df['scene'] = stats_df['filename'].str.replace(r'\.(mp4|avi|mov)$', '', regex=True)

# Only use features that are fully populated
features = [f for f in VIDEO_FEATURES if f in stats_df.columns
            and stats_df[f].notna().all()
            and (stats_df[f] != '').all()]

stats_df = stats_df[['scene'] + features]
print(f"Video stats: {len(stats_df)} videos, features: {features}")

Video stats: 20 videos, features: ['SI', 'TI', 'CF', 'TP', 'MV', 'DTP']


In [8]:
# ──────────────────────────────────────────────
# Cell 3: Compute sensitivity
# ──────────────────────────────────────────────

sensitivity_df = (
    df.groupby(['scene', 'parameter', 'resolution'])['score_centered']
    .agg(max_score='max', min_score='min')
    .reset_index()
)
sensitivity_df['sensitivity'] = sensitivity_df['max_score'] - sensitivity_df['min_score']

# Merge with video stats
analysis_df = sensitivity_df.merge(stats_df, on='scene', how='inner')
analysis_df['resolution'] = analysis_df['resolution'].astype(str)
sensitivity_df['resolution'] = sensitivity_df['resolution'].astype(str)

print(f"Analysis df: {len(analysis_df)} rows")
print(sensitivity_df.sort_values('sensitivity', ascending=False).head(10))


Analysis df: 304 rows
                      scene     parameter resolution  max_score  min_score  \
112                 oldmine  alpha_weight         50   8.760578 -16.108487   
64    fantasticvillage-open  alpha_weight         50   7.670465 -15.830397   
96       lightfoliage-close  alpha_weight         50   6.024611  -8.583810   
113                 oldmine  alpha_weight         71   4.754075  -7.063529   
65    fantasticvillage-open  alpha_weight         71   3.039592  -7.726338   
99       lightfoliage-close  alpha_weight        100   3.657008  -5.953572   
98       lightfoliage-close  alpha_weight         87   3.673240  -5.002206   
131              quarry-all  alpha_weight        100   3.463995  -4.800188   
97       lightfoliage-close  alpha_weight         71   3.851844  -4.211648   
272  wildwest-behindcounter  alpha_weight         50   2.811310  -5.175910   

     sensitivity  
112    24.869064  
64     23.500862  
96     14.608421  
113    11.817604  
65     10.765930  
99   

In [9]:
# ──────────────────────────────────────────────
# Cell 4: Heatmap — sensitivity per scene x TAA param per resolution
# ──────────────────────────────────────────────

def plot_sensitivity_heatmap(sensitivity_df):
    charts = []
    for res in RESOLUTIONS:
        df_res = sensitivity_df[sensitivity_df['resolution'] == res]
        if df_res.empty:
            continue
        chart = alt.Chart(df_res).mark_rect().encode(
            x=alt.X('parameter:N', title='TAA Parameter',
                    sort=PARAMS,
                    axis=alt.Axis(labelAngle=-30)),
            y=alt.Y('scene:N', title='Scene',
                    sort=alt.EncodingSortField(field='sensitivity', op='mean', order='descending')),
            color=alt.Color('sensitivity:Q',
                            scale=alt.Scale(scheme='viridis'),
                            title='Sensitivity (max-min)'),
            tooltip=['scene', 'parameter',
                     alt.Tooltip('sensitivity:Q', format='.3f'),
                     alt.Tooltip('max_score:Q', format='.3f'),
                     alt.Tooltip('min_score:Q', format='.3f')]
        ).properties(
            title=f'Resolution {res}%',
            width=300,
            height=380,
        )
        charts.append(chart)

    return alt.concat(*charts, columns=2).properties(
        title='TAA Parameter Sensitivity (centered CGVQM range)'
    ).configure_title(fontSize=13).configure_axis(labelFontSize=10, titleFontSize=11)

plot_sensitivity_heatmap(sensitivity_df)

alt.ConcatChart(...)

In [19]:
# add % label to resolution before plotting
analysis_df['resolution_label'] = analysis_df['resolution'].astype(str) + '%'
res_order  = ['100%', '87%', '71%', '50%']
res_colors = ['#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']
color_scale = alt.Scale(domain=res_order, range=res_colors)

def plot_sensitivity_vs_feature(analysis_df, features):
    # Melt features into long form
    id_cols = ['scene', 'resolution', 'resolution_label', 'parameter', 'sensitivity']
    melted = analysis_df[id_cols + features].melt(
        id_vars=id_cols,
        value_vars=features,
        var_name='feature',
        value_name='feature_value'
    )

    base = alt.Chart(melted).properties(width=150, height=130)

    points = base.mark_circle(size=45, opacity=0.9).encode(
        x=alt.X('feature_value:Q', title='',
                axis=alt.Axis(format='.3f', labelFontSize=9)),
        y=alt.Y('sensitivity:Q', title='Sensitivity'),
        color=alt.Color('resolution_label:N', scale=color_scale, sort=res_order, title='Resolution'),
        tooltip=['scene', 'resolution_label', 'parameter',
                 alt.Tooltip('feature_value:Q', format='.4f'),
                 alt.Tooltip('sensitivity:Q', format='.3f')]
    )

    regression = base.mark_line(strokeDash=[4, 2], opacity=0.5).transform_regression(
        'feature_value', 'sensitivity', groupby=['resolution_label']
    ).encode(
        x='feature_value:Q',
        y='sensitivity:Q',
        color=alt.Color('resolution_label:N', scale=color_scale, sort=res_order, title='Resolution'),
    )

    return (points + regression).facet(
        row=alt.Row('parameter:N', title='TAA Parameter',
                    sort=PARAMS,
                    header=alt.Header(labelAngle=0, labelAlign='left', labelFontSize=11)),
        column=alt.Column('feature:N', title='Video Feature',
                          sort=features,
                          header=alt.Header(labelAngle=-30, labelFontSize=11))
    ).resolve_scale(
        x='independent'
    ).properties(
        title='Video Characteristic vs TAA Sensitivity (by resolution)'
    ).configure_title(fontSize=13).configure_axis(labelFontSize=9, titleFontSize=11)

plot_sensitivity_vs_feature(analysis_df, features)

alt.FacetChart(...)

In [27]:
def plot_sensitivity_by_bin(analysis_df, feature, param=None, resolution=None):
    df = analysis_df.copy()
    df['resolution_label'] = df['resolution'].astype(str) + '%'
    df['feature_bin'] = pd.qcut(df[feature], q=3,
                                labels=[f'low {feature}', f'mid {feature}', f'high {feature}'])

    # Print bin membership
    bin_groups = df.drop_duplicates('scene').groupby('feature_bin')['scene']
    for bin_name, scenes in bin_groups:
        scene_list = sorted(scenes.tolist())
        print(f"{bin_name} ({len(scene_list)} videos): {', '.join(scene_list)}")
    print()
    
    if param:
        df = df[df['parameter'] == param]
    if resolution:
        df = df[df['resolution_label'] == resolution]

    res_order  = ['100%', '87%', '71%', '50%']
    res_colors = ['#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']
    color_scale = alt.Scale(domain=res_order, range=res_colors)

    chart = alt.Chart(df).mark_boxplot(extent='min-max').encode(
        x=alt.X('feature_bin:O', title=f'{feature} group',
                sort=[f'low {feature}', f'mid {feature}', f'high {feature}']),
        y=alt.Y('sensitivity:Q', title='Sensitivity (CGVQM range)'),
        color=alt.Color('resolution_label:N', scale=color_scale,
                        sort=res_order, title='Resolution'),
        tooltip=['scene', 'feature_bin',
                 alt.Tooltip('sensitivity:Q', format='.3f'),
                 alt.Tooltip(f'{feature}:Q', format='.4f')]
    ).properties(width=160, height=150)

    facet_kwargs = {}
    if not param:
        facet_kwargs['column'] = alt.Column('parameter:N', sort=PARAMS, title='TAA Parameter')
    if not resolution:
        facet_kwargs['row'] = alt.Row('resolution_label:N', sort=res_order)

    if facet_kwargs:
        chart = chart.facet(**facet_kwargs).resolve_scale(y='shared')

    return chart.properties(
        title=f'Sensitivity by {feature} group'
        + (f' | {param}' if param else '')
        + (f' | res {resolution}' if resolution else '')
    ).configure_title(fontSize=13).configure_axis(labelFontSize=10, titleFontSize=11)

# Example calls:
plot_sensitivity_by_bin(analysis_df, feature='MV')                          # all params, all resolutions


low MV (7 videos): abandoned, cubetest, lightfoliage-close, oldmine, resto-fwd, resto-pan, wildwest-town
mid MV (6 videos): abandoned-demo, quarry-all, resto-close, scifi, subway-lookdown, subway-turn
high MV (6 videos): abandoned-flipped, lightfoliage, quarry-rocksonly, wildwest-bar, wildwest-behindcounter, wildwest-store



/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_12904/132640114.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bin_groups = df.drop_duplicates('scene').groupby('feature_bin')['scene']


alt.FacetChart(...)

In [28]:
plot_sensitivity_by_bin(analysis_df, feature='SI')                          # all params, all resolutions


low SI (7 videos): abandoned, abandoned-demo, lightfoliage, resto-close, resto-fwd, resto-pan, subway-lookdown
mid SI (6 videos): cubetest, lightfoliage-close, oldmine, scifi, subway-turn, wildwest-behindcounter
high SI (6 videos): abandoned-flipped, quarry-all, quarry-rocksonly, wildwest-bar, wildwest-store, wildwest-town



/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_12904/132640114.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bin_groups = df.drop_duplicates('scene').groupby('feature_bin')['scene']


alt.FacetChart(...)

In [13]:
# ──────────────────────────────────────────────
# Cell 6: Score curves — CGVQM vs TAA parameter value
#         one line per scene, colored by a chosen video feature
#         adjust feature/param/resolution to explore
# ──────────────────────────────────────────────

def plot_score_curves(df, stats_df, feature, param, resolution, features):
    """
    Shows how centered CGVQM changes as a TAA parameter sweeps its values.
    Each line = one scene, colored by the chosen video feature.
    Useful for seeing if e.g. high-TI scenes have different curve shapes for alpha_weight.
    """
    if feature not in features:
        print(f"Feature '{feature}' not available. Choose from: {features}")
        return None

    df_plot = df[
        (df['parameter'] == param) &
        (df['resolution'] == int(resolution))
    ].merge(stats_df, on='scene', how='inner')

    return alt.Chart(df_plot).mark_line(point=True).encode(
        x=alt.X('value:Q', title=param),
        y=alt.Y('score_centered:Q', title='Centered CGVQM',
                scale=alt.Scale(zero=False)),
        color=alt.Color(f'{feature}:Q',
                        scale=alt.Scale(scheme='viridis'),
                        title=feature),
        detail='scene:N',
        tooltip=['scene',
                 alt.Tooltip('value:Q'),
                 alt.Tooltip('score_centered:Q', format='.3f'),
                 alt.Tooltip(f'{feature}:Q', format='.4f')]
    ).properties(
        title=f'Centered CGVQM vs {param} | res={resolution}% | colored by {feature}',
        width=600,
        height=380,
    ).configure_title(fontSize=13).configure_axis(labelFontSize=11, titleFontSize=12)

# Example — change feature/param/resolution to explore:
plot_score_curves(df, stats_df,
                  feature='SI',
                  param='alpha_weight',
                  resolution='100',
                  features=features)

alt.Chart(...)